# DX 704 Week 11 Project

In this project, you will develop and test prompts asking a language model to classify text from a home services query and match it to an appropriate category of home services.

The full project description and a template notebook are available on GitHub: [Project 11 Materials](https://github.com/bu-cds-dx704/dx704-project-11).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1 : Design a Short Prompt

The provided file "queries.txt" contains sample text from requests by homeowners by email or phone.
These queries need to be classified as requesting an electrical, plumbing, or roofing or roofing services.
The provided file has columns query_id, query, and target_category.
Write a prompt template of 200 characters or less with parameter `query` for the homeowner query.
Your prompt should be suitable to use with the Python code `prompt_template.format(query=query)`.
Test your prompt with the model `gpt-6-luna` and suitable parsing code.

In [ ]:
%pip install -U "openai>=3.26.0"

In [7]:
%pip install os

Note: you may need to restart the kernel to use updated packages.


c:\Users\caleb\Projects\BU Fall 2026\AI_in_the_Field_Coursework\field\Scripts\python.exe: No module named pip


In [18]:
# from google.colab import userdata
from openai import OpenAI
import os
import pandas as pd

In [12]:
client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

In [14]:
response = client.responses.create(
    model="gpt-6-luna",
    input="How much wood would a woodchuck chuck if a wood chuck could chuck wood?"
)

print(response.output_text)

The classic estimate is **about 700 pounds**—if a woodchuck could chuck wood, and if it worked as hard at wood as it does at digging.


In [21]:
queries = pd.read_csv('queries.txt', sep='\t')
queries.head(5)

,query_id,query,target_category
0,1,Hi. Melissa came by and wrecked my roof. Can y...,roofing
1,2,Hi there. This is Jack. I’m looking for someon...,plumbing
2,3,Can you install an automated spotlight by my d...,electrical
3,4,Pest control just cleared out a raccoon that t...,roofing
4,5,Need toilet unclogged ASAP,plumbing


In [70]:
short_prompt = """
Choose electrical, plumbing, or roofing (one word response and lowercase): {query}
"""
short_prompt

'\nChoose electrical, plumbing, or roofing (one word response and lowercase): {query}\n'

In [71]:
queries.loc[0]['query']

'Hi. Melissa came by and wrecked my roof. Can you take a look?'

In [72]:
response = client.responses.create(
    model="gpt-6-luna",
    input=short_prompt.format(query=queries.loc[0]['query'])
)

print(response.output_text)

roofing


Save your prompt template in a file "short-prompt.txt".
Save the results of your prompt testing in "short-output.tsv" with columns `query_id` and `predicted_category`.

In [73]:
with open("short-prompt.txt", "w", encoding="utf-8") as file:
    file.write(short_prompt)

In [74]:
short_output = pd.DataFrame({
    'query_id': [queries.loc[0]['query_id']],
    'predicted_category': [response.output_text]
})
short_output

,query_id,predicted_category
0,1,roofing


In [75]:
short_output.to_csv('short-output.tsv', index=False, sep='\t')

Submit "short-prompt.txt" and "short-output.tsv" in Gradescope.

Hint: your prompt may be re-tested with the Gemini API, so do not rely solely on lucky language model responses.

## Part 2: Find Short Prompt Mistakes

Construct 5 queries of 100 characters or less that trick your short prompt so that the wrong category is chosen.


In [76]:
def run_model(prompt: str):
    response = client.responses.create(
        model="gpt-6-luna",
        input=prompt
    )
    return response.output_text

In [119]:
difficult_queries = pd.DataFrame([
    # Leaky roof
    {
        'query': 'I found a leak in my bathroom coming down the wall',
        'target_category': 'roofing'
    },
    # Roof issue, light just not working as a result
    {
        'query': 'The light switch in the bathroom stopped working ever since we saw the crack in the roof',
        'target_category': 'roofing'
    },
    # There is a bathroom above with plumbing issues
    {
        'query': 'The ceiling has a water mark',
        'target_category': 'plumbing'
    },
    # Tree fall is more critical
    {
        'query': 'Winter storm resulted in leaks, frozen pipes, tree fall. Need immediate help',
        'target_category': 'roofing'
    },
    # Solar is the issue
    {
        'query': 'The new roof was installed with solar and now i notice electrical issues',
        'target_category': 'roofing'
    },
])
difficult_queries

,query,target_category
0,I found a leak in my bathroom coming down the ...,roofing
1,The light switch in the bathroom stopped worki...,roofing
2,The ceiling has a water mark,plumbing
3,"Winter storm resulted in leaks, frozen pipes, ...",roofing
4,The new roof was installed with solar and now ...,roofing


In [111]:
queries_copy = queries.copy()
queries_copy.head(3)

,query_id,query,target_category
0,1,Hi. Melissa came by and wrecked my roof. Can y...,roofing
1,2,Hi there. This is Jack. I’m looking for someon...,plumbing
2,3,Can you install an automated spotlight by my d...,electrical


In [112]:
def get_predicted_category(row):
    prompt = short_prompt.format(
        query=row['query']
    )
    return run_model(prompt)

queries_copy["predicted_category"] = queries_copy.apply(get_predicted_category, axis=1)
queries_copy.head(5)

,query_id,query,target_category,predicted_category
0,1,Hi. Melissa came by and wrecked my roof. Can y...,roofing,roofing
1,2,Hi there. This is Jack. I’m looking for someon...,plumbing,plumbing
2,3,Can you install an automated spotlight by my d...,electrical,electrical
3,4,Pest control just cleared out a raccoon that t...,roofing,roofing
4,5,Need toilet unclogged ASAP,plumbing,plumbing


In [113]:
queries_copy[queries_copy['target_category'] != queries_copy['predicted_category']]

,query_id,query,target_category,predicted_category


In [120]:
difficult_queries_copy = difficult_queries.copy()
difficult_queries_copy["predicted_category"] = difficult_queries_copy.apply(get_predicted_category, axis=1)
difficult_queries_copy.head(5)

,query,target_category,predicted_category
0,I found a leak in my bathroom coming down the ...,roofing,plumbing
1,The light switch in the bathroom stopped worki...,roofing,electrical
2,The ceiling has a water mark,plumbing,roofing
3,"Winter storm resulted in leaks, frozen pipes, ...",roofing,plumbing
4,The new roof was installed with solar and now ...,roofing,electrical


In [122]:
mistakes = difficult_queries_copy[difficult_queries_copy['target_category'] != difficult_queries_copy['predicted_category']]
mistakes

,query,target_category,predicted_category
0,I found a leak in my bathroom coming down the ...,roofing,plumbing
1,The light switch in the bathroom stopped worki...,roofing,electrical
2,The ceiling has a water mark,plumbing,roofing
3,"Winter storm resulted in leaks, frozen pipes, ...",roofing,plumbing
4,The new roof was installed with solar and now ...,roofing,electrical


Save your 5 queries in a file "mistakes.tsv" with columns `query`, `target_category` and `predicted_category`.

In [123]:
mistakes.to_csv('mistakes.tsv', index=False, sep='\t')

Submit "mistakes.tsv" in Gradescope.

## Part 3: Design a Long Prompt

Repeat part 1 with a length limit of 5000 characters.

In [124]:
longer_prompt = """
The following query contains text from requests by homeowners by email or phone.
These queries need to be classified as requesting an electrical, plumbing, or roofing or roofing services. Please only
respond with one of those categories.

Refer to the examples after the query which show you how to answer:

Query: {query}

Example 1. Query: Need a new electrical output. Output: electrical
Example 2. Query: Need my toilet fixed. Output: plumbing
Example 3. Query: Roof is leaking. Output: roofing
"""
longer_prompt

'\nThe following query contains text from requests by homeowners by email or phone.\nThese queries need to be classified as requesting an electrical, plumbing, or roofing or roofing services. Please only\nrespond with one of those categories.\n\nRefer to the examples after the query which show you how to answer:\n\nQuery: {query}\n\nExample 1. Query: Need a new electrical output. Output: electrical\nExample 2. Query: Need my toilet fixed. Output: plumbing\nExample 3. Query: Roof is leaking. Output: roofing\n'

Save your longer prompt template in a file "long-prompt.txt".
Save the results of your prompt testing in "long-output.tsv".
Both files should use the same columns as part 1.

In [125]:
with open("long-prompt.txt", "w", encoding="utf-8") as file:
    file.write(short_prompt)

In [126]:
response = client.responses.create(
    model="gpt-6-luna",
    input=longer_prompt.format(query=queries.loc[0]['query'])
)

print(response.output_text)

roofing


In [127]:
long_output = pd.DataFrame({
    'query_id': [queries.loc[0]['query_id']],
    'predicted_category': [response.output_text]
})
long_output
long_output.to_csv('long-output.tsv', index=False, sep='\t')

Submit "long-prompt.txt" and "long-output.tsv" in Gradescope.

## Part 4: Use the Decisions API

In September 2026, TypeSafe AI announced [Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev), a new variant of large language models focused on decision making.
In October 2026, OpenAI introduced a similar API that they called a [Decisions API](https://openai.com/index/decisions-api/).
Use the Decisions API ([docs](https://developers.openai.com/api/docs/guides/decisions)) to the implement the same decisions.
Save the results of your testing to "decision-output.tsv" with the same columns as parts 1 and 3.

In [133]:
response = client.decisions.create(
    model="gpt-6-luna",
    input=longer_prompt.format(query=queries.loc[0]['query']),
    questions=[
        {
            'type': 'choice',
            "instructions": "Sample text from requests by homeowners by email or phone. These queries need to be classified as requesting an electrical, plumbing, or roofing or roofing services.",
            'choices': [
                {"value": "electrical"},
                {"value": "roofing"},
                {"value": "plumbing"},
            ]
        }
    ]
)
response

Decision(answers=[AnswerAnswerResourceChoice(choice='roofing', confidence=0.99, name=None, probabilities=[AnswerAnswerResourceChoiceProbability(probability=0.0, value='electrical'), AnswerAnswerResourceChoiceProbability(probability=0.99, value='roofing'), AnswerAnswerResourceChoiceProbability(probability=0.01, value='plumbing')], type='choice')], model='gpt-6-luna', usage=Usage(input_tokens=257, input_tokens_details=UsageInputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=0, output_tokens_details=UsageOutputTokensDetails(reasoning_tokens=0), total_tokens=257))

In [137]:
response.answers[0].choice

'roofing'

In [138]:
decision_output = pd.DataFrame({
    'query_id': [queries.loc[0]['query_id']],
    'predicted_category': [response.answers[0].choice]
})
decision_output
decision_output.to_csv('decision-output.tsv', index=False, sep='\t')

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.